In [ ]:
# @title 0. Installing Dependencies and Setting up
# @markdown ### This Script Uses:
# @markdown - **PyTorch Lightning** – A high-level framework for PyTorch that simplifies training loops.
# @markdown - **Qubvel Segmentation Models (segmentation_models.pytorch)** – A collection of popular neural networks for semantic segmentation based on PyTorch.
# @markdown - **Albumentations** – A fast and flexible image augmentation library.
# @markdown - **Deepglobe Land Cover Dataset** – A fast and flexible image augmentation library.
# @markdown - **Gradio** – for GUI
# @markdown ---
# @markdown ### Dataset Requirements:
# @markdown To run this script properly, the **DeepGlobe Land Cover Classification Dataset** must be placed in your Google Drive.
# @markdown You can download it from [Kaggle](https://www.kaggle.com/datasets/balraj98/deepglobe-land-cover-classification-dataset). \
# @markdown Default dataset path: `drive/deepglobe`

!pip install -q git+https://github.com/PyTorchLightning/pytorch-lightning
!pip install -q git+https://github.com/qubvel/segmentation_models.pytorch
!pip install -q git+https://github.com/albumentations-team/albumentations
!pip install -q torchinfo
!pip install -q gradio

import os, torch, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pytorch_lightning as pl
from torch import nn
from tqdm.notebook import tqdm
from IPython.display import clear_output
from google.colab import drive
drive.mount('/content/drive')
torch.set_float32_matmul_precision('medium')

os.system('clear')
clear_output()
print("Setup Complete")

Setup Complete


In [ ]:
# @title 1. TRAINING CONSTANTS and image & mask fetch
# @markdown #### Params For Training:
# @markdown - **IMAGE SIZE** – 640
# @markdown - **BATCH SIZE** – 12
# @markdown - **MAX EPOCHS** – 50

# @markdown ---
# @markdown #### Dataset Size:
# @markdown - Training set size: 481
# @markdown - Validation set size: 121
# @markdown - Testing set size: 201
# @markdown - Total: 803

from glob import glob
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

IMAGE_SIZE = 640
BATCH_SIZE = 12
EPOCHS = 50

color_dict = pd.read_csv('../content/drive/MyDrive/deepglobe/class_dict.csv')
CLASSES = color_dict['name']

#drive data fetch
pd_dataset = pd.DataFrame({
    'IMAGES': sorted(glob("../content/drive/MyDrive/deepglobe/train/*.jpg")),
    'MASKS': sorted(glob("../content/drive/MyDrive/deepglobe/train/*.png"))
})
pd_dataset = shuffle(pd_dataset)
pd_dataset.reset_index(inplace=True, drop=True)
pd_dataset.head()

#dataset split train/test/val
pd_train, pd_test = train_test_split(pd_dataset, test_size=0.25, random_state=0)
pd_train, pd_val = train_test_split(pd_train, test_size=0.2, random_state=0)

os.system('clear')
clear_output()
print("image and mask connection/split done correctly")

image and mask connection/split done correctly


In [ ]:
# @title 2. RGB-MASK - CLASS-MASK CONVERSION | Training Configuration (scale/size etc)
import albumentations as aug

def rgb2category(rgb_mask): #converts RGB segmentation mask to integer labels.
    category_mask = np.zeros(rgb_mask.shape[:2], dtype=np.int8)
    for i, row in color_dict.iterrows():
        category_mask += (np.all(rgb_mask.reshape((-1, 3)) == (row['r'], row['g'], row['b']), axis=1).reshape(rgb_mask.shape[:2]) * i)
    return category_mask

def category2rgb(category_mask): #reverese it
    rgb_mask = np.zeros(category_mask.shape[:2] + (3,))
    for i, row in color_dict.iterrows():
        rgb_mask[category_mask==i] = (row['r'], row['g'], row['b'])
    return np.uint8(rgb_mask)

train_augment = aug.Compose([
    aug.Resize(IMAGE_SIZE, IMAGE_SIZE),
    aug.HorizontalFlip(p=0.5),
    aug.VerticalFlip(p=0.5),
    aug.RandomBrightnessContrast(p=0.3),
    aug.Affine(
    scale=(0.8, 1.0),
    translate_percent=0.0,
    rotate=0.0,
    p=0.5
    ),
    aug.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])


test_augment = aug.Compose([
    aug.Resize(IMAGE_SIZE, IMAGE_SIZE),
    aug.RandomBrightnessContrast(p=0.3),
    aug.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

In [ ]:
# @title 3. DATA LOADER MODULE
# @markdown Inside Module: Image reading | RGB → category label conversion | Augmentation | Tensor conversion | Data splitting | Batching via DataLoader then returns train, val, test dataloaders
from torch.utils.data import Dataset, DataLoader

class SegmentationDataset(Dataset):
    def __init__(self, df, augmentations=None):
        self.df = df
        self.augmentations = augmentations

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index: int):
        row = self.df.iloc[index]
        #Image reading
        image = cv2.imread(row.IMAGES)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        #mask reading
        mask = cv2.imread(row.MASKS)
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
        #Augmentation
        if self.augmentations:
            data = self.augmentations(image=image, mask=mask)
            image = data['image']
            mask = data['mask']

        mask = rgb2category(mask)
        image = np.transpose(image, (2, 0, 1)).astype(np.float64)
        mask = np.expand_dims(mask, axis=0)
        # Tensor conversion
        image = torch.Tensor(image) / 255.0
        mask = torch.Tensor(mask).long()

        return image, mask

class SegmentationDataModule(pl.LightningDataModule):
    def __init__(self, pd_train, pd_val, pd_test, batch_size=10):
        super().__init__() #Data splitting
        self.pd_train = pd_train
        self.pd_val = pd_val
        self.pd_test = pd_test
        self.batch_size=batch_size
    #Batching via DataLoader
    def setup(self, stage=None):
        self.train_dataset = SegmentationDataset(self.pd_train, train_augment)
        self.val_dataset = SegmentationDataset(self.pd_val, test_augment)
        self.test_dataset = SegmentationDataset(self.pd_test, test_augment)
    #Returns train, val, test dataloaders
    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=11,persistent_workers=True)
    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size // 2, shuffle=False, num_workers=11,persistent_workers=True)
    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size // 2, shuffle=False, num_workers=12,persistent_workers=True)


In [ ]:
# @title 4. Model Definition, Initlizing Unet++
from segmentation_models_pytorch import UnetPlusPlus
from segmentation_models_pytorch.losses import DiceLoss
from segmentation_models_pytorch.metrics import get_stats, iou_score, accuracy, precision, recall, f1_score

data_module = SegmentationDataModule(pd_train, pd_val, pd_test, batch_size=BATCH_SIZE)
data_module.setup()

class SegmentationModel(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = UnetPlusPlus(
            encoder_name="timm-regnety_120", #pretrained encoder
            encoder_weights="imagenet",
            in_channels=3,
            classes=len(CLASSES),
            activation="softmax"
        )
        self.criterion = DiceLoss(mode="multiclass", from_logits=False)

    def forward(self, inputs, targets=None):
        outputs = self.model(inputs)
        if targets is not None:
            loss = self.criterion(outputs, targets)
            tp, fp, fn, tn = get_stats(outputs.argmax(dim=1).unsqueeze(1).type(torch.int64), targets, mode='multiclass', num_classes=len(CLASSES))
            metrics = {
                "Accuracy": accuracy(tp, fp, fn, tn, reduction="micro-imagewise"),
                "IoU": iou_score(tp, fp, fn, tn, reduction="micro-imagewise"),
                "Precision": precision(tp, fp, fn, tn, reduction="micro-imagewise"),
                "Recall": recall(tp, fp, fn, tn, reduction="micro-imagewise"),
                "F1score": f1_score(tp, fp, fn, tn, reduction="micro-imagewise")
            }
            return loss, metrics, outputs
        else:
            return outputs

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=0.0001)

    def training_step(self, batch, batch_idx):
        images, masks = batch

        loss, metrics, outputs = self(images, masks)
        self.log_dict({
            "train/Loss": loss,
            "train/Accuracy": metrics['Accuracy'],
            "train/IoU": metrics['IoU'],
            "train/Precision": metrics['Precision'],
            "train/Recall": metrics['Recall'],
            "train/F1score": metrics['F1score']
        }, prog_bar=True, logger=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, masks = batch

        loss, metrics, outputs = self(images, masks)
        self.log_dict({
            "val/Loss": loss,
            "val/Accuracy": metrics['Accuracy'],
            "val/IoU": metrics['IoU'],
            "val/Precision": metrics['Precision'],
            "val/Recall": metrics['Recall'],
            "val/F1score": metrics['F1score']
        }, prog_bar=True, logger=True, on_step=False, on_epoch=True)
        return loss

    def test_step(self, batch, batch_idx):
        images, masks = batch

        loss, metrics, outputs = self(images, masks)
        self.log_dict({
            "test/Loss": loss,
            "test/Accuracy": metrics['Accuracy'],
            "test/IoU": metrics['IoU'],
            "test/Precision": metrics['Precision'],
            "test/Recall": metrics['Recall'],
            "test/F1score": metrics['F1score']
        }, prog_bar=True, logger=True, on_step=False, on_epoch=True)
        return loss
# model definition as var
model = SegmentationModel()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# @title 5. Checkpoint and Training Config. Module
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import CSVLogger

checkpoint_callback = ModelCheckpoint(
    dirpath="checkpoints",
    filename="best-checkpoint",
    save_top_k=1,
    verbose=True,
    monitor="val/F1score",
    mode="min"
)

#logging train data into csv file
logger = CSVLogger("lightning_logs", name="landcover-classification-log")

#early stop acording to Accuracy patience
early_stopping_callback = EarlyStopping(monitor="val/Accuracy", patience=4)

trainer = pl.Trainer(
    logger=logger,
    log_every_n_steps=31, # to minimize log flooding
    callbacks=[checkpoint_callback, early_stopping_callback], # calls back best trained epoch
    max_epochs=EPOCHS,
    accelerator="gpu",
    precision="16-mixed",
    devices=1
)

INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


In [ ]:
# @title 6. Train Module Call
trainer.fit(model, data_module)

/usr/local/lib/python3.11/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /content/checkpoints exists and is not empty.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name      | Type         | Params | Mode 
---------------------------------------------------
0 | model     | UnetPlusPlus | 71.0 M | train
1 | criterion | DiceLoss     | 0      | train
---------------------------------------------------
71.0 M    Trainable params
0         Non-trainable params
71.0 M    Total params
284.054   Total estimated model params size (MB)
655       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1739: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

INFO:pytorch_lightning.utilities.rank_zero:Epoch 2, global step 123: 'val/F1score' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:Epoch 3, global step 164: 'val/F1score' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:Epoch 4, global step 205: 'val/F1score' was not in top 1


In [ ]:
# @title 7. Test Module Call
trainer.test(model, data_module)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test/Accuracy       │    0.9475557208061218     │
│       test/F1score        │    0.8164454698562622     │
│         test/IoU          │     0.715211033821106     │
│         test/Loss         │    0.4802152216434479     │
│      test/Precision       │    0.8164454698562622     │
│        test/Recall        │    0.8164454698562622     │
└───────────────────────────┴───────────────────────────┘

[{'test/Loss': 0.4802152216434479,
  'test/Accuracy': 0.9475557208061218,
  'test/IoU': 0.715211033821106,
  'test/Precision': 0.8164454698562622,
  'test/Recall': 0.8164454698562622,
  'test/F1score': 0.8164454698562622}]

In [ ]:
# @title APP
import gradio as gr
import torch
import cv2
import numpy as np
from PIL import Image, ImageDraw

def predict_land_cover(uploaded_img):
    # Load and preprocess image
    image = np.array(uploaded_img.convert("RGB"))
    image_resized = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    input_tensor = torch.Tensor(image_resized.transpose(2, 0, 1)).unsqueeze(0) / 255.0

    # Predict
    with torch.no_grad():
        output = model(input_tensor)[0]
        pred_mask = output.argmax(dim=0).cpu().numpy()

    # Convert prediction to RGB
    rgb_mask = category2rgb(pred_mask)

    # Calculate class percentages
    unique, counts = np.unique(pred_mask, return_counts=True)
    percentages = {CLASSES[i]: round((counts[j] / pred_mask.size) * 100, 2) for j, i in enumerate(unique)}

    # Build legend image
    legend_height = len(percentages) * 30
    legend = Image.new("RGB", (300, legend_height), color="white")
    draw = ImageDraw.Draw(legend)

    for idx, (class_name, percent) in enumerate(percentages.items()):
        row = color_dict[color_dict['name'] == class_name].iloc[0]
        color = (int(row['r']), int(row['g']), int(row['b']))
        y = idx * 30
        draw.rectangle([0, y, 30, y + 30], fill=color)
        draw.text((35, y + 5), f"{class_name}: {percent}%", fill="black")

    return Image.fromarray(rgb_mask), legend

#Launch Gradio
gr.Interface(
    fn=predict_land_cover,
    inputs=gr.Image(type="pil"),
    outputs=[gr.Image(type="pil", label="Segmentation Mask"), gr.Image(type="pil", label="Legend")],
    title="Land Cover Segmentation",
    description="Upload a satellite image to see class-wise percentages and segmentation mask"
).launch(debug=True)

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2bd49b62af83f93a97.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1739: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1739: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)
